# Deep Agent RAG Demo

Chat with the PDF documents in a Dalux file area or folder, using a LangChain `deepagents` deep agent, OpenRouter as the model provider, and a local Chroma vector index.

**Prerequisites**
- `pip install 'dalux-build[rag]'` (or `uv sync --extra rag` in `python/`)
- `git`, `yarn`, and the LangGraph CLI (`langgraph`) on your `PATH`
- `DALUX_BASE_URL` / `DALUX_API_KEY` / `OPENROUTER_API_KEY` set (e.g. in the repo's `.env`)

See `python/dalux_build/ai/agent/` for the implementation and `python/scripts/skills.sh` for the skills-library search CLI.

## Setup

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
python_dir = project_root / "python"

if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))
    print(f"✓ Added {python_dir} to Python path")

import os
from dotenv import load_dotenv

load_dotenv()

required = ["DALUX_BASE_URL", "DALUX_API_KEY", "OPENROUTER_API_KEY"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    print(f"⚠️  Missing environment variables: {', '.join(missing)}")
else:
    print("✓ All required environment variables are set")

✓ Added /Users/brunoadam/Documents/development/github/dalux-build/python/notebooks/python to Python path
✓ All required environment variables are set


In [2]:
from dalux_build import create_client

dalux = create_client()

projects = dalux.projects.list_projects()
if len(projects) == 1:
    PROJECT_ID = projects[0].project_id
else:
    PROJECT_ID = os.getenv("DALUX_PROJECT_ID")
dalux.set_default_project(PROJECT_ID)
print(f"✓ Using project ID: {PROJECT_ID}")

✓ Using project ID: S313578016888324096


/var/folders/d1/x6l5g3q17xl7sxts9r454k6r0000gn/T/ipykernel_46787/649189243.py:5: DeprecationWarning: list_projects() is deprecated and only returns the first page of results. Use get_projects() instead to fetch all projects with pagination.
  projects = dalux.projects.list_projects()


## Browse file areas

`dalux.ai.file_areas.agent()` and `dalux.ai.files.agent()` both scope to a file area — the former to the whole area, the latter to a specific folder within it (via `folder_id` or `path`).

In [3]:
file_areas = dalux.file_areas.get_file_areas()
file_areas

[FileArea(file_area_id='S313578021116182528', file_area_name='Files', file_area_type='files'),
 FileArea(file_area_id='S313578021132959744', file_area_name='Shared files', file_area_type='shared'),
 FileArea(file_area_id='S313578021149736960', file_area_name='Published files', file_area_type='published')]

## Search the skills library

Skills are `SKILL.md` files (the Agent Skills standard) under `dalux_build/ai/agent/skills/`. `search_skills` is the same search `python/scripts/skills.sh` uses from the shell.

In [4]:
from dalux_build.ai.skills_cli import search_skills

for skill in search_skills("legal"):
    print(f"{skill.name}: {skill.description}")

entrepriseret: Danish construction contract law (entrepriseret) — AB18, ABT18, ABR18 standard terms, mangler, forsinkelse/dagbod, voldgift. Use when the user asks about Danish construction contracts, AB18/ABT18/ABR18, or entrepriseretlige spørgsmål.
legal-contract-review: Reads and answers questions about legal contractual documents (clauses, obligations, deadlines, liabilities, termination terms) in construction projects. Use when the user asks about a contract, agreement, terms, obligations, or liability.


## Launch a RAG chat agent over a folder of PDFs

This will:
1. List the PDFs in the given folder (recursively by default) and download any new/changed ones into a local cache.
2. Chunk and embed them into a local, per-scope Chroma index (re-embeds only what changed on repeat runs).
3. Start a local LangGraph backend (`langgraph dev`) serving a `deepagents` agent over that index.
4. Clone (first run only) and start `deep-agents-ui`, the browser chat frontend, and open it in your browser.

First run downloads a local embeddings model and clones/installs `deep-agents-ui`, so it can take a few minutes. `deep-agents-ui` doesn't support pre-filling its connection settings via URL, so paste the printed **Deployment URL** and **Assistant ID** into its settings dialog once it opens.

In [5]:
dalux.set_default_file_area(file_area_id=file_areas[0].file_area_id)

In [6]:
!uv sync --extra rag -qU

In [7]:
handle = dalux.ai.files.agent(
    path="Files/1_Archive/A6_Tender project",  # or folder_id="..."
    skill="legal-contract-review",
    verbose=True,
    provider="mistral",
)

print(f"Chat UI:       {handle.ui_url}")
print(f"Deployment URL: {handle.backend_url}")
print(f"Assistant ID:   {handle.assistant_id}")

GET /6.1/projects/S313578016888324096/file_areas/S313578021116182528/files params={'includeProperties': True}


Fetching pages: 100%|██████████| 5169/5169 [00:12<00:00, 424.17item/s]


Files matching folder 'S315408395299454976': 837
Files matching filter: 732 / 837
Found 422 PDF(s) in scope.


Syncing PDFs: 100%|██████████| 422/422 [00:00<00:00, 3246.28file/s, LLYN.B250_K40_C10.10_ZRTM.00026.01_ABA ABDL.pdf]                                                                                      
/Users/brunoadam/Documents/development/github/dalux-build/python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading local embeddings model BAAI/bge-small-en-v1.5 (downloads on first use, this can take a few minutes)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9502.45it/s]


Embeddings model loaded.
No new or changed PDFs to embed.
Starting langgraph dev backend on http://127.0.0.1:53192 (cwd=/Users/brunoadam/Library/Caches/dalux-build/rag/c75e102385b554db/langgraph_run)...
Backend ready.
Preparing deep-agents-ui in /Users/brunoadam/Library/Caches/dalux-build/rag/_shared/deep-agents-ui (clones/installs on first run)...
Starting deep-agents-ui on http://localhost:53258...
Chat UI ready.



Deep agent chat UI starting.
  Open: http://localhost:53258
  In the settings dialog, enter:
    Deployment URL: http://127.0.0.1:53192
    Assistant ID:   dalux_agent



Chat UI:       http://localhost:53258
Deployment URL: http://127.0.0.1:53192
Assistant ID:   dalux_agent


## Choosing a model

By default `.agent()` uses OpenRouter (`OPENROUTER_API_KEY`, default model `z-ai/glm-5.2:free`). Two ways to change that:
- Pass `model="<openrouter-model-id>"` directly.
- Set `pick_model=True` (optionally `free_only=True`) for an interactive numbered picker, or call `pick_openrouter_model()` yourself.

Or switch provider entirely with `provider="mistral"` (uses `MISTRAL_API_KEY`, default model `mistral-large-latest`).

In [ ]:
from dalux_build.ai.agent import pick_openrouter_model

# Prints a numbered menu of OpenRouter models (optionally free-tier only)
# and prompts for a choice; returns the chosen model id.
# picked_model = pick_openrouter_model(free_only=True)
# handle = dalux.ai.files.agent(path="Files/1_Archive/A6_Tender project/C02_Agreement", model=picked_model)

# Or let .agent() do the picking for you in one call:
# handle = dalux.ai.files.agent(
#     path="Files/1_Archive/A6_Tender project/C02_Agreement",
#     pick_model=True,
#     free_only=True,
# )

In [ ]:
# handle = dalux.ai.files.agent(
#     path="Files/1_Archive/A6_Tender project/C02_Agreement",
#     skill="legal-contract-review",
#     provider="mistral",  # uses MISTRAL_API_KEY, model defaults to "mistral-large-latest"
#     verbose=True,
# )

## Reference corpus: Danish building regulations (byggerietsregler.dk)

A standing reference corpus, independent of any Dalux project — crawled once via [Firecrawl](https://firecrawl.dev) into a themed markdown sitemap, then indexed into its own persistent Chroma collection. Once indexed, **every** `.agent()` call automatically gets a `search_byggerietsregler` tool alongside `search_dalux_documents` — no extra wiring needed per call. Re-crawling to refresh is a separate, manual step.

Requires `FIRECRAWL_API_KEY` in `.env` (get one at [firecrawl.dev](https://firecrawl.dev)).

In [ ]:
from dalux_build.ai.reference import reference_store_exists

print("byggerietsregler indexed:", reference_store_exists("byggerietsregler"))

# One-time (or occasional refresh) operation — crawls the whole site via
# Firecrawl, saves it as a themed markdown sitemap under
# ~/Library/Caches/dalux-build/reference/byggerietsregler/markdown/, and
# indexes it into its own persistent Chroma collection. Requires
# FIRECRAWL_API_KEY in .env. This can take a long time (blocks until the
# whole site is crawled) and consumes Firecrawl credits (1 per page).
#
from dalux_build.ai.reference import crawl_and_index_byggerietsregler
crawl_and_index_byggerietsregler(verbose=True)

## Live web search

Separate from the standing reference corpus above: a `web_search` tool (via Firecrawl's search API) is added to every agent automatically whenever `FIRECRAWL_API_KEY` is set — no extra setup. It searches the live web and scrapes matching pages as markdown, for specific laws or publicly available documents that aren't in the indexed Dalux documents or reference corpora. Nothing to run here; it just becomes available once the key is in `.env`.

### Scoping to an entire file area instead of one folder

In [ ]:
# handle = dalux.ai.file_areas.agent(
#     file_area_id=file_areas[0].file_area_id,
#     skill="legal-contract-review",
# )

## Stop the agent

Both local processes (the LangGraph backend and the chat UI) keep running until stopped explicitly.

In [ ]:
handle.stop()